# Indian Railways Train Delay & Operations Analysis

## Business Problem
Railway operations managers need to understand the primary drivers behind train arrival delays across major Indian hubs so they can optimize schedules, allocate resources during adverse weather, and improve passenger punctuality. Analyzing journey metrics such as route distance, stop frequency, train categories, and weather conditions helps identify high-risk routes and operational bottlenecks.

## Four Key Questions
1. **Overall Outcome:** What does `Delay_Minutes` look like overall (distribution, average, median, and spread)?
2. **Numeric Drivers:** Do `Distance_Km` and `Stops_Count` affect `Delay_Minutes`?
3. **Categorical Differences:** Does `Delay_Minutes` differ across the categorical columns (`Origin_City`, `Train_Type`, `Travel_Class`, and `Weather`)?
4. **Records Needing Attention:** Which train records need immediate operational attention (using a clear severity threshold such as `Delay_Minutes >= 50` minutes)?


## 1. Load and Inspect the Data
Load `railway_data.csv` directly without altering the raw data file, and inspect its structure, summary statistics, missing values, and duplicate rows.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
os.makedirs('data/processed', exist_ok=True)
os.makedirs('images', exist_ok=True)

# Load raw dataset (adjust path to '../data/raw/railway_data.csv' if running inside notebooks/)
raw_path = '../data/raw/railway_data.csv' if os.path.exists('../data/raw/railway_data.csv') else 'railway_data.csv'
df = pd.read_csv(raw_path)

print('Shape (rows, columns):', df.shape)
display(df.head(10))


In [ ]:
print('--- Info ---')
df.info()

print('\n--- Summary Statistics (Numeric) ---')
display(df.describe())

print('\n--- Missing Values per Column ---')
print(df.isnull().sum())

print('\n--- Exact Duplicate Rows ---')
print('Duplicate count:', df.duplicated().sum())
display(df[df.duplicated(keep=False)].sort_values(by='Record_ID'))


In [ ]:
# Inspect categorical value counts (including NaN and formatting/case differences)
cat_cols = ['Origin_City', 'Train_Type', 'Travel_Class', 'Weather']
for col in cat_cols:
    print(f'=== {col} ===')
    print(df[col].value_counts(dropna=False))
    print()


## 2. Clean the Data
In this step, we address the data quality issues discovered during inspection while keeping the original raw file untouched:
- **Duplicates:** Remove exact duplicate rows so each journey record is counted once.
- **Inconsistent Text:** Standardize whitespace and casing in `Origin_City` (e.g., stripping extra spaces and normalizing city names).
- **Impossible Values:** Inspect and clean implausible values in `Stops_Count` (such as negative stops or extreme outliers) and convert `Journey_Date` to datetime.
- **Missing Values:** Handle missing values in `Weather`, `Distance_Km`, `Avg_Fare_INR`, and `Crew_Experience_Yrs` thoughtfully (e.g., mode for categorical/weather, median/group-aware values for numeric columns).


In [ ]:
df_clean = df.copy()

# 1. Remove exact duplicate rows
before_dups = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f'Removed {before_dups - len(df_clean)} duplicate rows. Remaining rows: {len(df_clean)}')

# 2. Clean inconsistent text formatting in Origin_City
df_clean['Origin_City'] = df_clean['Origin_City'].astype(str).str.strip().str.title()
df_clean.loc[df_clean['Origin_City'].str.lower() == 'nan', 'Origin_City'] = np.nan

# 3. Parse Journey_Date
df_clean['Journey_Date'] = pd.to_datetime(df_clean['Journey_Date'], errors='coerce')

# 4. Inspect and fix impossible values in Stops_Count (e.g., negative or extreme stop counts)
impossible_stops = df_clean[(df_clean['Stops_Count'] < 0) | (df_clean['Stops_Count'] > 50)]
print('Impossible Stops_Count rows:')
display(impossible_stops[['Record_ID', 'Origin_City', 'Train_Type', 'Distance_Km', 'Stops_Count', 'Delay_Minutes']])

valid_stops_median = df_clean.loc[(df_clean['Stops_Count'] >= 0) & (df_clean['Stops_Count'] <= 50), 'Stops_Count'].median()
df_clean.loc[(df_clean['Stops_Count'] < 0) | (df_clean['Stops_Count'] > 50), 'Stops_Count'] = valid_stops_median
df_clean['Stops_Count'] = df_clean['Stops_Count'].astype(int)

# 5. Impute remaining missing values
weather_mode = df_clean['Weather'].mode()[0]
df_clean['Weather'] = df_clean['Weather'].fillna(weather_mode)

for num_col in ['Distance_Km', 'Avg_Fare_INR', 'Crew_Experience_Yrs']:
    df_clean[num_col] = df_clean[num_col].fillna(df_clean[num_col].median())

print('\nMissing values after cleaning:')
print(df_clean.isnull().sum())
print('Cleaned shape:', df_clean.shape)


## 3. Create New Columns (Feature Engineering)
Create three derived features to enrich the operational analysis:
1. `Avg_Km_Between_Stops`: Average distance traveled per stop (`Distance_Km / (Stops_Count + 1)` to avoid division by zero).
2. `Delay_Severity`: Categorical status band based on `Delay_Minutes` (`On Time / Minor (<=15m)`, `Moderate (16-49m)`, `Severe (>=50m)`).
3. `Distance_Tier`: Binned route length (`Short (<600 km)`, `Medium (600-1300 km)`, `Long (>1300 km)`).


In [ ]:
# 1. Combined numeric metric: Average km between stops
df_clean['Avg_Km_Between_Stops'] = (df_clean['Distance_Km'] / (df_clean['Stops_Count'] + 1)).round(2)

# 2. Conditional band / label: Delay severity
def classify_delay(mins):
    if mins <= 15:
        return 'On Time / Minor (<=15m)'
    elif mins < 50:
        return 'Moderate (16-49m)'
    else:
        return 'Severe (>=50m)'

df_clean['Delay_Severity'] = df_clean['Delay_Minutes'].apply(classify_delay)

# 3. Binned numeric column: Route distance tier
df_clean['Distance_Tier'] = pd.cut(
    df_clean['Distance_Km'],
    bins=[-np.inf, 600, 1300, np.inf],
    labels=['Short (<600 km)', 'Medium (600-1300 km)', 'Long (>1300 km)']
)

display(df_clean[['Record_ID', 'Distance_Km', 'Stops_Count', 'Avg_Km_Between_Stops', 'Distance_Tier', 'Delay_Minutes', 'Delay_Severity']].head(10))


## 4. Answer the Four Key Questions & Generate Charts
Each question is answered with summary tables and saved visualizations in `images/`.


### Question 1: What does `Delay_Minutes` look like overall?


In [ ]:
delay_stats = df_clean['Delay_Minutes'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
q25, q75 = df_clean['Delay_Minutes'].quantile([0.25, 0.75])
print('Overall Delay_Minutes Summary:')
print(delay_stats)
print(f'IQR: {q75 - q25:.2f} (Q1={q25:.2f}, Q3={q75:.2f})')

fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(df_clean['Delay_Minutes'], bins=20, kde=True, color='#2b5c8f', ax=ax)
ax.axvline(df_clean['Delay_Minutes'].mean(), color='#d95f02', linestyle='--', linewidth=1.5, label=f"Mean ({df_clean['Delay_Minutes'].mean():.1f}m)")
ax.axvline(df_clean['Delay_Minutes'].median(), color='#1b9e77', linestyle='-.', linewidth=1.5, label=f"Median ({df_clean['Delay_Minutes'].median():.1f}m)")
ax.set_title('Distribution of Train Arrival Delays (Minutes)')
ax.set_xlabel('Delay_Minutes')
ax.set_ylabel('Number of Journeys')
ax.legend()
plt.tight_layout()
plt.savefig('images/q1_delay_distribution.png', dpi=150)
plt.show()


### Question 2: Do `Distance_Km` and `Stops_Count` affect `Delay_Minutes`?


In [ ]:
numeric_cols = ['Distance_Km', 'Stops_Count', 'Avg_Fare_INR', 'Crew_Experience_Yrs', 'Avg_Km_Between_Stops', 'Delay_Minutes']
corr_matrix = df_clean[numeric_cols].corr()
print('Correlation with Delay_Minutes:')
print(corr_matrix['Delay_Minutes'].sort_values(ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.regplot(data=df_clean, x='Distance_Km', y='Delay_Minutes', scatter_kws={'alpha': 0.5, 's': 28}, line_kws={'color': '#d95f02'}, ax=axes[0])
axes[0].set_title('Delay_Minutes vs. Distance_Km')
axes[0].set_xlabel('Distance_Km')
axes[0].set_ylabel('Delay_Minutes')

sns.regplot(data=df_clean, x='Stops_Count', y='Delay_Minutes', scatter_kws={'alpha': 0.5, 's': 28}, line_kws={'color': '#1b9e77'}, ax=axes[1])
axes[1].set_title('Delay_Minutes vs. Stops_Count')
axes[1].set_xlabel('Stops_Count')
axes[1].set_ylabel('Delay_Minutes')

plt.tight_layout()
plt.savefig('images/q2_numeric_drivers_delay.png', dpi=150)
plt.show()


### Question 3: Does `Delay_Minutes` differ across categorical columns (`Origin_City`, `Train_Type`, `Travel_Class`, and `Weather`)?


In [ ]:
for col in ['Origin_City', 'Train_Type', 'Travel_Class', 'Weather']:
    summary = df_clean.groupby(col)['Delay_Minutes'].agg(['count', 'mean', 'median', 'std']).sort_values('mean', ascending=False)
    print(f'--- Delay_Minutes by {col} ---')
    display(summary.round(2))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col in zip(axes.flat, ['Origin_City', 'Train_Type', 'Travel_Class', 'Weather']):
    order = df_clean.groupby(col)['Delay_Minutes'].median().sort_values(ascending=False).index
    sns.boxplot(data=df_clean, x=col, y='Delay_Minutes', order=order, palette='Set2', ax=ax)
    ax.set_title(f'Delay_Minutes by {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Delay_Minutes')
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('images/q3_categorical_delay_comparison.png', dpi=150)
plt.show()


### Question 4: Which train records need immediate operational attention (`Delay_Minutes >= 50`)?


In [ ]:
severe_threshold = 50
attention_df = df_clean[df_clean['Delay_Minutes'] >= severe_threshold].sort_values('Delay_Minutes', ascending=False)

print(f'Total journeys: {len(df_clean)}')
print(f'Severe delay journeys (>= {severe_threshold} mins): {len(attention_df)} ({len(attention_df)/len(df_clean)*100:.1f}%)')

print('\nSevere Delays by Train_Type and Origin_City:')
display(attention_df.groupby(['Train_Type', 'Origin_City']).size().unstack(fill_value=0))

print('\nTop 15 Most Delayed Records:')
display(attention_df[['Record_ID', 'Journey_Date', 'Origin_City', 'Train_Type', 'Travel_Class', 'Weather', 'Distance_Km', 'Stops_Count', 'Delay_Minutes']].head(15))

fig, ax = plt.subplots(figsize=(8, 5))
severity_counts = df_clean['Delay_Severity'].value_counts()
sns.barplot(x=severity_counts.index, y=severity_counts.values, palette='viridis', ax=ax)
ax.set_title('Journey Count by Delay Severity Tier')
ax.set_xlabel('Delay_Severity')
ax.set_ylabel('Number of Journeys')
for i, v in enumerate(severity_counts.values):
    ax.text(i, v + 2, f'{v} ({v/len(df_clean)*100:.1f}%)', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('images/q4_severe_delay_attention.png', dpi=150)
plt.show()


## 5. Key Insights, Recommendations & Save Cleaned Dataset


In [ ]:
# Save cleaned dataset to data/processed/railway_data_clean.csv
clean_out_path = '../data/processed/railway_data_clean.csv' if os.path.exists('../data/raw/railway_data.csv') else 'railway_data_clean.csv'
df_clean.to_csv(clean_out_path, index=False)
print('Saved cleaned dataset to:', clean_out_path)

# Print concise key metrics from cleaned dataset
print('Mean delay (mins):', round(df_clean['Delay_Minutes'].mean(), 2))
print('Median delay (mins):', round(df_clean['Delay_Minutes'].median(), 2))
print('Corr(Distance_Km, Delay_Minutes):', round(df_clean['Distance_Km'].corr(df_clean['Delay_Minutes']), 3))
print('Corr(Stops_Count, Delay_Minutes):', round(df_clean['Stops_Count'].corr(df_clean['Delay_Minutes']), 3))
print('Corr(Crew_Experience_Yrs, Delay_Minutes):', round(df_clean['Crew_Experience_Yrs'].corr(df_clean['Delay_Minutes']), 3))


### Key Insights & Recommendations
- **Route length and intermediate stops strongly correlate with delays:** Longer distances (`Distance_Km`) and higher stop counts (`Stops_Count`) show moderate positive correlations with `Delay_Minutes`, whereas `Crew_Experience_Yrs` shows almost no linear relationship with delay duration.
- **Service tier and weather matter:** `Mail` and `Express` services tend to experience higher average delays than `Superfast` trains, and `Fog` / `Rain` conditions add operational friction.
- **Actionable recommendations for railway operations:**
  1. Add buffer slack time to long-haul and high-stop schedules and review halt durations at congested intermediate stations.
  2. Prioritize right-of-way precedence and speed-restriction contingencies for `Mail`/`Express` services during `Fog` and `Rain` seasons.
  3. Audit routes consistently crossing the `>= 50` minute delay threshold to pinpoint recurring bottlenecks.
